## Reflection with LangChain (Tweet Generator)

### Generate

In [1]:
pip install --upgrade -q openai langchain langchain-openai langchain-community langgraph

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

#### What is Reflection Pattern?

Reflection pattern is where an LLm acts like a Human doing meditation.  
Agent generates a thought or a response and then it reflects on it and generates a better response everytime it reflects.  

In [4]:
from langchain_core.messages import AIMessage, HumanMessage, BaseMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI

In [6]:
# Generation Prompt
generation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            'system',
            '''You are a Twitter expert assigned to craft outstanding tweets.
            Generate the most engaging and impactful tweet possible based on the user's request.
            If the user provides feedback, refine and enhance your previous attempts accordingly for maximum engagement.'''
        ),
        MessagesPlaceholder(variable_name="messages") # Dynamic placeholder which saves all the reflections on the agent
    ]
)

llm = ChatOpenAI(model_name='gpt-4o-mini', temperature=0.7)

# using LCEL to create the generate_chain
generate_chain = generation_prompt | llm

In [7]:
generate_chain

ChatPromptTemplate(input_variables=['messages'], input_types={'messages': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_core.mes

In [8]:
tweet = ''

request = HumanMessage(
    content="FIFA World Cup 26"
)

for chunk in generate_chain.stream({'messages': [request]}):
    print(chunk.content, end='')
    tweet += chunk.content

🏆🌍 Get ready for the ultimate showdown! The #FIFAWorldCup26 is coming to North America! 🇺🇸🇨🇦🇲🇽 Which team are you cheering for? 🤔⚽ Let the countdown to glory begin! ⏳🔥 #WorldCup #Soccer #FootballFever

### Reflect and Repeat

In [10]:
reflection_prompt = ChatPromptTemplate.from_messages(
    [
        (
            'system',
            '''You are a Twitter influencer known for your engaging content and sharp insights.
            Review and critique the user's tweet.
            Provide constructive feedback, focusing on enhancing its depth, style and overall impact.
            Offer specific suggestions to make the tweet more compelling and engaging for their audience.
            '''
        ),
        MessagesPlaceholder(variable_name='messages'),
    ]
)

reflection_chain = reflection_prompt | llm



In [11]:
reflection = ''

for chunk in reflection_chain.stream({'messages': [request, HumanMessage(content=tweet)]}):
    print(chunk.content, end='')
    reflection += chunk.content

This tweet has a strong foundation with great enthusiasm and the use of emojis to capture attention. However, it could be enhanced to make an even greater impact. Here are some suggestions:

1. **Add a Personal Touch**: Share a personal insight or experience related to the World Cup. This helps create a connection with your audience. For example, you could mention a memorable moment from a past World Cup or why this tournament is particularly exciting for you.

2. **Engage with a Question**: Instead of a general question like "Which team are you cheering for?", consider a more specific or provocative question that could spark discussion. For example, "Which underdog team do you think could surprise everyone this year?" This invites varied responses and can lead to richer engagement.

3. **Include a Call to Action**: Encourage your audience to share their thoughts or predictions. A simple "Reply with your dream match-up!" can increase interaction.

4. **Enhance Visual Appeal**: While em

In [12]:
# Use the reflection and the generated initial tweet to get a better tweet
for chunk in generate_chain.stream(
    { 'messages': [request, AIMessage(content=tweet), HumanMessage(content=reflection)]}
):
    print(chunk.content, end='')

🏆🌍 The countdown to #FIFAWorldCup26 is on, and I can’t contain my excitement! 🇺🇸🇨🇦🇲🇽 My favorite moment? That last-minute goal in 2014! 🙌⚽ What’s yours? This year, I’m backing [Your Team]! Who do you think will be the surprise contender? Share your dream match-up below! ⏳🔥 #WorldCup #Soccer #FootballFever #DreamMatch